# Two-Day Microstructure Alpha Pipeline - Preliminary Research

**Hypothesis:** Tomorrow's high/low "invade" yesterday's high/low in a way that is predictable, conditioned on volatility regime. Two core signals:

- **Downward invasion (dhl):** `H_t - L_{t+1}` normalized by `H_t - L_t`
- **Upward invasion (dlh):** `H_{t+1} - L_t` normalized by `H_t - L_t`

**Outcomes tested:** next-day candle direction (green/red), next-day range expansion, next-day wick structure, next-day gap direction.

**Regime conditioning:** VIX < 15 (Low), 15-25 (Mid), > 25 (High).

This notebook covers data preparation, variable selection, model selection with QA (ADF-stationary residuals), conditional probability tables, monotonicity tests, and time-stability segmentation. Backtesting is intentionally excluded (research phase only).


## 1. Data Pull

Daily OHLCV for QQQ (Nasdaq-100 ETF, inception 1999) and the VIX index (volatility regime driver). We pull from 2006 onward so all three VIX regimes are well represented. Daily frequency is what the microstructure hypothesis operates on (yesterday vs. tomorrow extremes).


In [ ]:
qb = QuantBook()
qqq = qb.add_equity("QQQ", Resolution.DAILY)
vix = qb.add_index("VIX", Resolution.DAILY)

start_date = datetime(2006, 1, 1)
end_date = datetime(2026, 9, 30)

qqq_history = qb.history(qqq.symbol, start_date, end_date, Resolution.DAILY)
vix_history = qb.history(vix.symbol, start_date, end_date, Resolution.DAILY)

qqq_df = qqq_history.droplevel(0)[["open", "high", "low", "close", "volume"]]
vix_df = vix_history.droplevel(0)[["close"]].rename(columns={"close": "vix"})
# Bar close timestamps differ between equities (16:00) and indices (15:15): normalize to calendar dates
qqq_df.index = pd.to_datetime(qqq_df.index.date)
vix_df.index = pd.to_datetime(vix_df.index.date)
qqq_df.index.name = "date"
vix_df.index.name = "date"

print("QQQ rows:", len(qqq_df), "| VIX rows:", len(vix_df))
print("QQQ range:", qqq_df.index.min(), "to", qqq_df.index.max())
print("VIX range:", vix_df.index.min(), "to", vix_df.index.max())
print("Median bar spacing (QQQ):", qqq_df.index.to_series().diff().median())
print("Median bar spacing (VIX):", vix_df.index.to_series().diff().median())
qqq_df.head()


## 2. Data Type and Signal Construction

**Data type:** daily OHLCV bar data (price extremes, range, volume) plus an implied-volatility index series. All series are positive-valued time series with trading-calendar gaps (holidays), so `infer_freq` returns None but the median spacing is 1 trading day.

**Signals (known only after t+1 close, used as features/targets for next-day outcomes):**

- `dhl_raw = H_t - L_{t+1}` (downward invasion), `dlh_raw = H_{t+1} - L_t` (upward invasion)
- `dhl_norm = dhl_raw / range_t`, `dlh_norm = dlh_raw / range_t` (scale-free)

**Targets (outcomes at t+1):** `dir_next` (green candle = 1), `range_ratio` (range_{t+1}/range_t), `gap_next` (open_{t+1}/close_t - 1), `wick_up_next`, `wick_down_next`.

**Features known at t:** prior-day range, ATR(14), VIX, regime label, and lagged invasion signals. We also add first differences, seasonal differences (lags 2, 5, 10, 21) and logs per the standard transform suite.


In [ ]:

px = qqq_df.copy()
px = px.join(vix_df, how="inner")
# Remove only clearly corrupted bars (range > 50x trailing median; keeps flash-crash days intact)
px["range"] = px["high"] - px["low"]
bad_dates = px.index[px["range"] > 50 * px["range"].rolling(63, min_periods=5).median()]
print("Dropped outlier bars:", list(bad_dates))
px = px.drop(index=bad_dates)
px["atr14"] = px["range"].rolling(14).mean()

# Two core microstructure signals (shift -1 aligns t+1 extremes onto row t)
px["dhl_raw"] = px["high"] - px["low"].shift(-1)
px["dlh_raw"] = px["high"].shift(-1) - px["low"]
px["dhl_norm"] = px["dhl_raw"] / px["range"]
px["dlh_norm"] = px["dlh_raw"] / px["range"]

# Outcomes at t+1
px["dir_next"] = (px["close"].shift(-1) > px["open"].shift(-1)).astype(float)
px["range_ratio"] = px["range"].shift(-1) / px["range"]
px["gap_next"] = px["open"].shift(-1) / px["close"] - 1.0
px["wick_up_next"] = (px["high"].shift(-1) - np.maximum(px["open"].shift(-1), px["close"].shift(-1))) / px["range"].shift(-1)
px["wick_down_next"] = (np.minimum(px["open"].shift(-1), px["close"].shift(-1)) - px["low"].shift(-1)) / px["range"].shift(-1)
px["range_expansion"] = (px["range_ratio"] > 1.0).astype(float)
px["gap_up"] = (px["gap_next"] > 0).astype(float)

# Regime from same-day VIX (usable: VIX close is known at t close for next-day decisions)
px["regime"] = pd.cut(px["vix"], bins=[-np.inf, 15, 25, np.inf], labels=["Low", "Mid", "High"])

# Features known at t
px["ret_1d"] = px["close"].pct_change()
px["vol_log"] = np.log(px["volume"])
px["vix_chg"] = px["vix"].diff()
px["dhl_lag"] = px["dhl_norm"].shift(1)
px["dlh_lag"] = px["dlh_norm"].shift(1)
px["dir_lag"] = px["dir_next"].shift(1)

# Differencing, seasonal differencing (lags 2, 5, 10, 21), log transforms
core_series = ["close", "range", "vix", "dhl_norm", "dlh_norm"]
for col in core_series:
    px[f"{col}_diff"] = px[col].diff()
    for lag in [2, 5, 10, 21]:
        px[f"{col}_sd{lag}"] = px[col].diff(lag)
    if (px[col] > 0).all():
        px[f"{col}_log"] = np.log(px[col])

px = px.dropna(subset=["dhl_norm", "dlh_norm", "dir_next", "range_ratio", "gap_next"])

print("Total data points (rows):", len(px))
print("Median bar spacing:", px.index.to_series().diff().median())
print("\nMissing data points per column (top 15):")
missing = px.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).head(15))
print("Total missing cells:", int(px.isna().sum().sum()))
print("\nRegime counts:\n", px["regime"].value_counts())


## 3. Summary Metrics

Mean, variance, skewness, and kurtosis for the core signals, targets, and transformed series. Heavy right tails in volume and range (log transforms tame these); normalized invasion signals are bounded roughly by construction.


In [ ]:
stat_cols = ["dhl_norm", "dlh_norm", "dhl_raw", "dlh_raw", "range", "vix", "volume",
             "ret_1d", "range_ratio", "gap_next", "wick_up_next", "wick_down_next",
             "dir_next", "range_expansion", "gap_up",
             "close_diff", "close_sd21", "vix_diff", "vix_sd21", "dhl_norm_diff", "dlh_norm_diff",
             "range_log", "vix_log", "close_log", "vol_log"]
stats_df = pd.DataFrame({
    "mean": px[stat_cols].mean(),
    "variance": px[stat_cols].var(),
    "skewness": px[stat_cols].skew(),
    "kurtosis": px[stat_cols].kurt(),
})
print(stats_df.round(4).to_string())


In [ ]:
px["range"].nlargest(10)


In [ ]:
plot_cols = ["close", "range", "vix", "dhl_norm", "dlh_norm",
             "close_log", "range_log", "vix_log", "dhl_norm_diff", "dlh_norm_diff",
             "close_sd21", "range_sd21", "vix_sd21", "dhl_norm_sd21", "dlh_norm_sd21",
             "dir_next", "range_expansion", "gap_up", "wick_up_next", "wick_down_next"]
n_cols = 4
n_rows = int(np.ceil(len(plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
for ax, col in zip(axes.flatten(), plot_cols):
    ax.plot(px.index, px[col], lw=0.6)
    ax.set_title(col)
for ax in axes.flatten()[len(plot_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


## 4. Variable Selection

Leakage control: models may only use information available at the **close of day t** (the decision time). Contemporaneous invasion signals (`dhl_norm`, `dlh_norm`) are diagnostic only - they measure the *structure* of the t/t+1 relationship. The tradeable features are lagged invasions (`dhl_lag`, `dlh_lag`), regime, and momentum/vol transforms.

Selection: rank by Pearson/Spearman/Kendall correlation against `dir_next`, take top 5; then fit a decision tree and take its top 5 by importance.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

px["atr_ratio"] = px["atr14"] / px["close"]
px["range_norm"] = px["range"] / px["close"]
px["vol_log_diff"] = px["vol_log"].diff()
# LEAK-SAFE invasion transforms: built from dhl_norm.shift(1), fully known at the t close
for col in ["dhl_norm", "dlh_norm"]:
    px[f"{col}_lag_diff"] = px[col].shift(1).diff()
    for lag in [5, 10, 21]:
        px[f"{col}_lag_sd{lag}"] = px[col].shift(1) - px[col].shift(1 + lag)

feature_set = ["dhl_lag", "dlh_lag", "dir_lag", "ret_1d", "atr_ratio", "range_norm", "vix", "vix_chg",
               "vix_sd5", "vix_sd10", "vix_sd21", "close_sd5", "close_sd10", "close_sd21",
               "range_sd5", "range_sd10", "range_sd21", "dhl_norm_lag_diff", "dhl_norm_lag_sd5",
               "dhl_norm_lag_sd10", "dhl_norm_lag_sd21", "dlh_norm_lag_diff", "dlh_norm_lag_sd5",
               "dlh_norm_lag_sd10", "dlh_norm_lag_sd21", "vol_log_diff"]
assert not any("_diff" in f and "lag" not in f and f not in ["vix_chg", "vol_log_diff", "close_sd5", "close_sd10", "close_sd21", "range_sd5", "range_sd10", "range_sd21", "vix_sd5", "vix_sd10", "vix_sd21"] for f in feature_set)
model_frame = px[feature_set + ["dir_next", "range_expansion", "gap_up", "regime"]].dropna()
print("Model frame rows:", len(model_frame))

corr_rows = []
for feat in feature_set:
    pear = model_frame[feat].corr(model_frame["dir_next"], method="pearson")
    spear = model_frame[feat].corr(model_frame["dir_next"], method="spearman")
    kend = model_frame[feat].corr(model_frame["dir_next"], method="kendall")
    corr_rows.append((feat, pear, spear, kend, np.mean([abs(pear), abs(spear), abs(kend)])))
corr_table = pd.DataFrame(corr_rows, columns=["feature", "pearson", "spearman", "kendall", "mean_abs"]).set_index("feature")
top5_corr = corr_table.sort_values("mean_abs", ascending=False).head(5)
print("\nTop 5 by mean absolute correlation:")
print(top5_corr.round(4).to_string())

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=7)
tree.fit(model_frame[feature_set], model_frame["dir_next"])
importances = pd.Series(tree.feature_importances_, index=feature_set)
top5_tree = importances.sort_values(ascending=False).head(5)
print("\nTop 5 by decision tree importance:")
print(top5_tree.round(4).to_string())

selected_5 = list(dict.fromkeys(top5_corr.index.tolist() + top5_tree.index.tolist()))[:5]
print("\nSelected top-5 feature set:", selected_5)


## 5. Model Selection and QA

Five candidate models predict next-day direction (`dir_next`) from the selected features, evaluated with **expanding-window walk-forward** (train on all data before each of 5 test blocks). QA: residuals must be ADF stationary using the **no-constant, no-trend** specification (`adfuller(resid, regression='n')`), and accuracy must beat the majority-class baseline. If no model passes with 5 variables, we widen to 20 and retry once, per the QA protocol.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from statsmodels.tsa.stattools import adfuller, acf, pacf

model_factories = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=2000),
    "DecisionTree": lambda: DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=7),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=100, random_state=7, n_jobs=-1),
    "GradientBoosting": lambda: GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=7),
    "GaussianNB": lambda: GaussianNB(),
}

def evaluate_models(model_frame, features, target, factories, n_blocks=5):
    data = model_frame[features + [target]].dropna()
    split_points = np.linspace(int(len(data) * 0.4), len(data) - 20, n_blocks + 1, dtype=int)
    results = []
    for name, factory in factories.items():
        preds = pd.Series(np.nan, index=data.index)
        for i in range(n_blocks):
            train_end, test_end = split_points[i], split_points[i + 1]
            model = factory()
            model.fit(data[features].iloc[:train_end], data[target].iloc[:train_end])
            preds.iloc[train_end:test_end] = model.predict(data[features].iloc[train_end:test_end])
        mask = preds.notna()
        y_true = data[target][mask]
        y_pred = preds[mask]
        accuracy = float((y_true == y_pred).mean())
        baseline = float(max(y_true.mean(), 1 - y_true.mean()))
        resid = y_true - y_pred
        adf_stat, adf_p = adfuller(resid, regression="n")[:2]
        results.append({"model": name, "accuracy": accuracy, "baseline": baseline,
                        "edge": accuracy - baseline, "adf_stat": adf_stat, "adf_p": adf_p,
                        "stationary": bool(adf_p < 0.05), "n_test": int(mask.sum())})
    return pd.DataFrame(results).set_index("model"), data, preds


In [ ]:
pass1_features = selected_5
results_5, data_5, preds_5 = evaluate_models(model_frame, pass1_features, "dir_next", model_factories)
print("=== QA Pass 1: top-5 variables ===")
print(results_5.round(4).to_string())

good = results_5[(results_5["accuracy"] > results_5["baseline"] + 0.02) & results_5["stationary"]]
if len(good) == 0:
    print("\nNo model passed QA with 5 variables. Expanding to 20 variables and retrying.")
    pass2_features = corr_table.sort_values("mean_abs", ascending=False).head(20).index.tolist()
    results_20, data_20, preds_20 = evaluate_models(model_frame, pass2_features, "dir_next", model_factories)
    print("\n=== QA Pass 2: top-20 variables ===")
    print(results_20.round(4).to_string())
    final_results = results_20
else:
    print("\nModels passing QA:", list(good.index))
    final_results = results_5


### Leakage fix

The first feature set produced a suspicious ~64% directional accuracy. Diagnosis: `dhl_norm_diff = dhl_norm_t - dhl_norm_{t-1}` and all seasonal-difference versions still embed `L_{t+1}` (tomorrow's low) because `dhl_norm_t = (H_t - L_{t+1}) / range_t`. Any feature built from the **unshifted** invasion signal leaks the target. The correlation selector happily picked these. Fix: every invasion-derived feature must be built from `dhl_norm.shift(1)`, which is fully known at the t close. The contemporaneous signals stay in scope only for the diagnostic conditional-probability analysis in Section 7.


## 6. Reporting: Final Three Models

Per the QA protocol the selection looped twice (top-5, then top-20 variables) with **no model achieving good accuracy** - every model underperforms the 54.1% majority baseline. Residuals are ADF stationary in all cases, so the failures are not a non-stationarity artifact: there is simply no learnable next-day directional signal in these features. Below we report the top three models by accuracy, their residual diagnostics (ACF/PACF, ADF no-constant/no-trend), and accuracy-vs-residual plots.


In [ ]:

top3 = final_results.sort_values("accuracy", ascending=False).head(3)
top3_names = top3.index.tolist()
print("Top three models:", top3_names)

features_used = pass2_features if "results_20" in dir() else pass1_features
data_eval = model_frame[features_used + ["dir_next"]].dropna()
split_points = np.linspace(int(len(data_eval) * 0.4), len(data_eval) - 20, 6, dtype=int)

residual_store = {}
for name in top3_names:
    factory = model_factories[name]
    preds = pd.Series(np.nan, index=data_eval.index)
    for i in range(5):
        model = factory()
        model.fit(data_eval[features_used].iloc[:split_points[i]], data_eval["dir_next"].iloc[:split_points[i]])
        preds.iloc[split_points[i]:split_points[i + 1]] = model.predict(data_eval[features_used].iloc[split_points[i]:split_points[i + 1]])
    residual_store[name] = (data_eval["dir_next"] - preds).dropna()

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
for row, name in enumerate(top3_names):
    resid = residual_store[name]
    acc = final_results.loc[name, "accuracy"]
    axes[row, 0].plot(resid.index, resid.values, lw=0.4)
    axes[row, 0].set_title(f"{name} residuals (acc={acc:.3f})")
    axes[row, 1].plot(np.cumsum(resid.values), lw=0.8, color="tab:orange")
    axes[row, 1].set_title(f"{name} cumulative residual error")
    lags = 20
    acf_vals = acf(resid, nlags=lags, fft=False)
    pacf_vals = pacf(resid, nlags=lags, method="ywm")
    conf = 1.96 / np.sqrt(len(resid))
    axes[row, 2].bar(range(lags + 1), acf_vals, width=0.3, label="ACF")
    axes[row, 2].bar(np.arange(lags + 1) + 0.35, pacf_vals, width=0.3, label="PACF")
    axes[row, 2].axhline(conf, color="r", ls="--", lw=0.8)
    axes[row, 2].axhline(-conf, color="r", ls="--", lw=0.8)
    axes[row, 2].set_title(f"{name} residual ACF/PACF")
    axes[row, 2].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("\nADF test on residuals (no constant, no trend, regression='n'):")
for name in top3_names:
    resid = residual_store[name]
    adf_out = adfuller(resid, regression="n")
    print(f"{name}: stat={adf_out[0]:.4f}, p={adf_out[1]:.5f}, lags={adf_out[2]}, nobs={adf_out[3]}, crit_1pct={adf_out[4]['1%']:.4f}")


## 7. Conditional Probability Tables and Monotonicity (Diagnostic)

These tests use the **contemporaneous** invasion signals (`dhl_norm`, `dlh_norm`) - they measure whether tomorrow's outcome is structurally linked to how tomorrow's extremes invade yesterday's range. This is a *structure* test: an edge that lives here could only be harvested with intraday information (e.g., entering when price is near yesterday's high/low), not at yesterday's close. Quantile bins per regime, plus logistic slope/significance per regime.


In [ ]:
from statsmodels.api import Logit, add_constant

cpt_frame = px[["dhl_norm", "dlh_norm", "dir_next", "range_expansion", "gap_up", "regime"]].dropna()
for outcome in ["dir_next", "range_expansion", "gap_up"]:
    print(f"\n================ P({outcome}) by dhl_norm quintile within regime ================")
    tables = []
    for reg in ["Low", "Mid", "High"]:
        sub = cpt_frame[cpt_frame["regime"] == reg]
        bins = pd.qcut(sub["dhl_norm"], 5, labels=False, duplicates="drop")
        prob = sub[outcome].groupby(bins).mean()
        count = sub[outcome].groupby(bins).size()
        tables.append(pd.DataFrame({reg: prob.round(3), reg + "_n": count}))
    print(pd.concat(tables, axis=1).to_string())

print("\n================ Logistic monotonicity: outcome ~ signal, per regime ================")
for signal in ["dhl_norm", "dlh_norm"]:
    for outcome in ["dir_next", "range_expansion"]:
        print(f"\n{outcome} ~ {signal}")
        for reg in ["Low", "Mid", "High"]:
            sub = cpt_frame[cpt_frame["regime"] == reg]
            model = Logit(sub[outcome], add_constant(sub[[signal]])).fit(disp=0)
            print(f"  {reg:4s}: slope={model.params[signal]:+.4f}  p={model.pvalues[signal]:.4f}  pseudoR2={model.prsquared:.4f}  AIC={model.aic:.1f}")


In [ ]:

px["gap_open_next"] = px["gap_next"]
stab_frame = px[["dhl_norm", "dlh_norm", "gap_open_next", "dir_next", "range_expansion", "regime"]].dropna()

def segment_stability(frame, signal, outcome, n_segments):
    seg_len = len(frame) // n_segments
    rows = []
    for i in range(n_segments):
        sub = frame.iloc[i * seg_len:(i + 1) * seg_len]
        model = Logit(sub[outcome], add_constant(sub[[signal]])).fit(disp=0)
        rows.append({"segment": i, "start": sub.index.min().date(), "end": sub.index.max().date(),
                     "slope": model.params[signal], "p": model.pvalues[signal], "n": len(sub)})
    return pd.DataFrame(rows)

print("=== Time stability: slope of dir_next ~ dhl_norm across segment counts ===")
for n_seg in [3, 5, 7, 10]:
    table = segment_stability(stab_frame, "dhl_norm", "dir_next", n_seg)
    signs = set(np.sign(table["slope"]))
    print(f"\n{n_seg} segments (sign consistent: {signs == {np.sign(table['slope'].iloc[0]) and -1.0 or signs}}):")
    print(table.round(4).to_string(index=False))

print("\n=== Tradeable proxy: P(dir_next) by gap sign at t+1 open, per regime ===")
stab_frame["gap_sign"] = np.sign(stab_frame["gap_open_next"])
print(stab_frame.groupby(["regime", "gap_sign"])["dir_next"].agg(["mean", "count"]).round(3).to_string())

print("\n=== Day-of-week check for the dhl_norm signal (slope stability) ===")
stab_frame["dow"] = stab_frame.index.day_name()
for dow in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
    sub = stab_frame[stab_frame["dow"] == dow]
    model = Logit(sub["dir_next"], add_constant(sub[["dhl_norm"]])).fit(disp=0)
    print(f"{dow:10s}: slope={model.params['dhl_norm']:+.4f}  p={model.pvalues['dhl_norm']:.4f}  n={len(sub)}")


In [ ]:
print("=== Performance metrics: top three models (QA pass 2, 20 variables) ===")
print(top3.round(4).to_string())
print("\nNotes: edge = accuracy - majority baseline. All residuals ADF stationary (regression='n'),")
print("but edge is negative for every model, so no directional alpha is extractable at the prior close.")


## 8. Conclusions and Assumptions

**What the data says:**

1. **The structural relation is real and extremely stable.** `dhl_norm` (tomorrow's low vs yesterday's high, range-normalized) predicts tomorrow's candle direction, range expansion, and gap direction with p < 0.0001, monotonic quintile behavior, consistent slope sign across 3/5/7/10 time segments (2006-2026) and all five weekdays, and consistent behavior in all three VIX regimes. Slopes barely decay in recent segments (-0.93 to -1.4).

2. **But the edge is not available at yesterday's close.** The signal contains `L_{t+1}` by construction - a tautology of candle anatomy (a low above yesterday's high requires a gap-up that holds). Every leak-safe model (lagged invasions, VIX regime, ATR, momentum, volume transforms - 5 variables, then 20) *underperformed* the 54.1% majority baseline in expanding-window walk-forward. All residuals were ADF stationary, so the null result is not a stationarity artifact: there is no learnable close-of-day-t directional signal.

3. **The open-time proxy also fails.** Conditioning only on the t+1 gap sign (known at the open) yields P(green) of 51.6-54.6% - indistinguishable from baseline. The predictive power of `dhl_norm` comes from where the day's *low* ends up, i.e. intraday information.

**Recommendation for the quant developer:** do not build a prior-close directional strategy from this signal. The viable research direction is **intraday conditional logic**: at time-of-touch during day t+1, the running `dhl` (distance from yesterday's high to the intraday low) is observable and the conditional probability tables in Section 7 apply directly. An intraday variant (e.g., trade only when price reclaims/holds yesterday's extremes, regime-conditioned, time-stamped exits) is the natural follow-up. Range-expansion prediction via `dlh_norm`/`dhl_norm` is the strongest structural result (pseudo-R2 up to 0.14) and may be useful for position sizing rather than direction.

**Assumptions made:** VIX close at t is usable for next-day regime labeling; the 2007-01-22 corrupted bar (range ~$3,697, ~90x the trailing median) was dropped; classification residuals are defined as y_true - y_pred in {-1, 0, 1}; walk-forward uses 5 expanding test blocks over the last 60% of data; no transaction costs are modeled in this research phase (backtesting is out of scope here).


## 9. Where Does the Structure Live? Regime and Time-Bucket Concentration

Quantifying which VIX regime and which time periods carry the most explanatory power (pseudo-R2 and likelihood-ratio chi2), and what that implies for alpha horizon and position sizing.


In [ ]:
from scipy.stats import chi2 as chi2_dist

def signal_strength(frame, signal, outcome):
    model = Logit(frame[outcome], add_constant(frame[[signal]])).fit(disp=0)
    llr = 2 * (model.llf - model.llnull)
    return {"slope": model.params[signal], "p": model.pvalues[signal],
            "pseudo_r2": model.prsquared, "llr_chi2": llr,
            "chi2_per_obs": llr / len(frame), "n": len(frame)}

print("=== Regime concentration: dir_next ~ dhl_norm ===")
regime_rows = []
for reg in ["Low", "Mid", "High"]:
    sub = stab_frame[stab_frame["regime"] == reg]
    row = signal_strength(sub, "dhl_norm", "dir_next")
    row["regime"] = reg
    row["share_of_days"] = len(sub) / len(stab_frame)
    regime_rows.append(row)
regime_table = pd.DataFrame(regime_rows).set_index("regime")
regime_table["contribution_share"] = regime_table["llr_chi2"] / regime_table["llr_chi2"].sum()
print(regime_table.round(4).to_string())

print("\n=== Regime concentration: range_expansion ~ dlh_norm ===")
for reg in ["Low", "Mid", "High"]:
    sub = stab_frame[stab_frame["regime"] == reg]
    row = signal_strength(sub, "dlh_norm", "range_expansion")
    print(f"{reg:4s}: pseudo_r2={row['pseudo_r2']:.4f}  chi2/obs={row['chi2_per_obs']:.5f}  n={row['n']}")

print("\n=== Yearly time buckets: dir_next ~ dhl_norm ===")
stab_frame["year"] = stab_frame.index.year
year_rows = []
for year, sub in stab_frame.groupby("year"):
    row = signal_strength(sub, "dhl_norm", "dir_next")
    row["year"] = year
    year_rows.append(row)
year_table = pd.DataFrame(year_rows).set_index("year")
year_table["chi2_share"] = year_table["llr_chi2"] / year_table["llr_chi2"].sum()
print(year_table.round(4).to_string())
print("\nStrongest 3 years by chi2:", year_table["llr_chi2"].nlargest(3).index.tolist())
print("Weakest 3 years by chi2:", year_table["llr_chi2"].nsmallest(3).index.tolist())
print("Last 3 years avg pseudo-R2:", year_table["pseudo_r2"].tail(3).mean().round(4))


### Section 9 answers

**Regime concentration (chi2 per observation, direction ~ dhl_norm):** Mid VIX (15-25) carries the structure most densely (0.176), High VIX nearly as strong (0.170), Low VIX weakest (0.153). Upward invasion (dlh_norm) strength *rises* with VIX (0.111 High > 0.099 Mid > 0.076 Low), while range-expansion prediction via dhl_norm is strongest in Low (0.138) and High (0.132) and weakest in Mid (0.107). Gap-direction quality degrades in High VIX (top-quintile P(gap up) = 0.841 vs 0.91 in Low/Mid).

**Time-bucket concentration (annual chi2 per observation):** crisis/vol-expansion years dominate - 2014 (0.270), 2022 (0.254), 2009 (0.231), 2008 (0.208), 2011 (0.203). Calm grind years are weakest - 2015 (0.105), 2025 (0.111), 2019 (0.121), 2013 (0.122). Recent years run 0.11-0.18, below the long-run mean of ~0.17 but nowhere near zero: the structure is volatility-cyclical, not decaying.

**Implications for alpha horizon:** the edge is *event-driven and intraday*, not steady-state. It concentrates in elevated-vol periods, so activity should be conditioned on the live VIX/realized-vol level, not the calendar. Decision resolution must be intraday (time-of-touch at yesterday's extremes); the holding horizon is hours-to-close, not multi-day.

**Implications for sizing:** scale exposure to regime edge density - full directional size in Mid/High VIX, reduced size in Low VIX where the directional structure is weakest (but use Low-VIX periods for the range-expansion/ATR-size information instead). Within High VIX, per-day variance roughly offsets the higher signal density, so ATR-normalized sizing with wider stops is required; the degraded gap quality in High VIX argues for stops based on yesterday's extremes rather than gap-based fills.


## 10. Implementation Inputs for the Intraday Rule

Three things the intraday rule needs that we have not yet pinned down: (1) the historical quintile cut points per regime for classifying running invasions, (2) the joint dhl x dlh structure - what happens when both triggers fire in the same day (Trigger A vs B conflict, Exit B validation), and (3) intraday first-touch timing: when during the day does the first extreme touch occur, and what does the running invasion look like at that moment. Items 1-2 use daily data; item 3 uses minute bars for the most recent two years.


In [ ]:

print("=== Quintile cut points of dhl_norm / dlh_norm by regime (implementation thresholds) ===")
for reg in ["Low", "Mid", "High"]:
    sub = px[px["regime"] == reg]
    cuts_dhl = sub["dhl_norm"].quantile([0.2, 0.4, 0.6, 0.8]).round(3).tolist()
    cuts_dlh = sub["dlh_norm"].quantile([0.2, 0.4, 0.6, 0.8]).round(3).tolist()
    print(f"{reg:4s} dhl_norm cuts @20/40/60/80%: {cuts_dhl}   dlh_norm cuts: {cuts_dlh}")

print("\n=== Joint table: P(dir_next) by dhl quintile (rows) x dlh quintile (cols) ===")
joint = px[["dhl_norm", "dlh_norm", "dir_next"]].dropna().copy()
joint["dhl_q"] = pd.qcut(joint["dhl_norm"], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
joint["dlh_q"] = pd.qcut(joint["dlh_norm"], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
print(joint.groupby(["dhl_q", "dlh_q"])["dir_next"].agg(["mean", "count"]).unstack()["mean"].round(3).to_string())

print("\n=== Trigger overlap: P(both invasions occur in same day) and conflict frequency ===")
both = ((px["dhl_norm"] > 0) & (px["dlh_norm"] > 0)).mean()
print(f"P(both downward and upward invasion happen in the same day): {both:.3f}")
q_num = joint["dhl_q"].cat.codes - joint["dlh_q"].cat.codes
print("Distribution of dhl_q - dlh_q (0 = same quintile class = rule conflict):")
print(q_num.value_counts().sort_index().to_string())


In [ ]:

minute_history = qb.history(qqq.symbol, datetime(2024, 10, 1), datetime(2026, 9, 29), Resolution.MINUTE)
minute_df = minute_history.droplevel(0)[["open", "high", "low"]]
minute_df["date"] = pd.to_datetime(minute_df.index.date)
print("Minute bars:", len(minute_df), "| days:", minute_df["date"].nunique())
print("First bar time:", minute_df.index.min().time(), "| Last bar time:", minute_df.index.max().time())
print(minute_df.head(3).to_string())


In [ ]:

daily_px = px[["high", "low", "range", "regime", "dhl_norm"]]
touch_rows = []
for date, day_bars in minute_df.groupby("date"):
    if date not in daily_px.index:
        continue
    prev = daily_px.loc[:date].iloc[-2]
    if prev is None or pd.isna(prev["range"]) or prev["range"] <= 0:
        continue
    day_dir = float(qqq_df.loc[date, "close"] > qqq_df.loc[date, "open"])
    running_low = day_bars["low"].cummin()
    running_high = day_bars["high"].cummax()
    down_touch = day_bars.index[running_low <= prev["low"]]
    up_touch = day_bars.index[running_high >= prev["high"]]
    t_down = down_touch[0] if len(down_touch) else None
    t_up = up_touch[0] if len(up_touch) else None
    first_t = min(t_down, t_up) if (t_down and t_up) else (t_down or t_up)
    if first_t is None:
        continue
    i = day_bars.index.get_loc(first_t)
    running_at_touch = pd.Series({
        "touch_time": first_t.time(),
        "touch_type": "down" if (t_down and (t_up is None or t_down <= t_up)) else "up",
        "minutes_to_close": len(day_bars) - i - 1,
        "dhl_running": (prev["high"] - running_low.iloc[i]) / prev["range"],
        "dlh_running": (running_high.iloc[i] - prev["low"]) / prev["range"],
        "day_dir": day_dir,
        "regime": daily_px.loc[date, "regime"],
    })
    touch_rows.append(running_at_touch)

touch_df = pd.DataFrame(touch_rows)
touch_df["touch_hour"] = pd.to_datetime(touch_df["touch_time"], format="%H:%M:%S").dt.hour
print("Days with a first touch analyzed:", len(touch_df))
print("\n=== First-touch time distribution (hour of day) ===")
print(touch_df["touch_hour"].value_counts().sort_index().to_string())
print("\nMedian minutes from first touch to close:", touch_df["minutes_to_close"].median())
print("P(first touch after 14:30):", round(float((touch_df["touch_time"] >= pd.to_datetime("14:30").time()).mean()), 3))

mid_cuts = px[px["regime"] == "Mid"]["dhl_norm"].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
down_touch = touch_df[touch_df["touch_type"] == "down"]
print("\n=== P(day ends green | running dhl_norm quintile at first downward touch) ===")
down_binned = pd.cut(down_touch["dhl_running"], [0] + mid_cuts + [np.inf], labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
print(down_touch["day_dir"].groupby(down_binned, observed=True).agg(["mean", "count"]).round(3).to_string())

print("\n=== P(day ends green) by first-touch type and hour bucket ===")
touch_df["hour_bucket"] = pd.cut(touch_df["touch_hour"], [8, 10, 12, 14, 16], labels=["09-11", "11-13", "13-15", "15-16"])
print(touch_df.groupby(["touch_type", "hour_bucket"], observed=True)["day_dir"].agg(["mean", "count"]).round(3).to_string())


In [ ]:

checkpoint_rows = []
for date, day_bars in minute_df.groupby("date"):
    if date not in daily_px.index:
        continue
    prev = daily_px.loc[:date].iloc[-2]
    if prev is None or pd.isna(prev["range"]) or prev["range"] <= 0:
        continue
    day_dir = float(qqq_df.loc[date, "close"] > qqq_df.loc[date, "open"])
    for check_time in ["12:00", "13:00", "14:00"]:
        sub = day_bars[day_bars.index.time <= pd.to_datetime(check_time).time()]
        if len(sub) < 30:
            continue
        checkpoint_rows.append({
            "date": date, "checkpoint": check_time,
            "dhl_running": (prev["high"] - sub["low"].min()) / prev["range"],
            "dlh_running": (sub["high"].max() - prev["low"]) / prev["range"],
            "day_dir": day_dir,
            "regime": daily_px.loc[date, "regime"],
        })

check_df = pd.DataFrame(checkpoint_rows)
for cp in ["12:00", "13:00", "14:00"]:
    sub = check_df[check_df["checkpoint"] == cp]
    binned = pd.cut(sub["dhl_running"], [0] + mid_cuts + [np.inf], labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    table = sub["day_dir"].groupby(binned, observed=True).agg(["mean", "count"])
    print(f"\n=== P(day ends green | running dhl_norm at {cp}) ===")
    print(table.round(3).to_string())

cp14 = check_df[check_df["checkpoint"] == "14:00"]
model = Logit(cp14["day_dir"], add_constant(cp14[["dhl_running"]])).fit(disp=0)
print(f"\n14:00 checkpoint logit: slope={model.params['dhl_running']:+.3f}  p={model.pvalues['dhl_running']:.4f}  pseudoR2={model.prsquared:.4f}  n={len(cp14)}")


### Section 10 findings (implementation inputs for the intraday rule)

**1. Quintile thresholds (Mid-regime, full history):** dhl_norm cuts at 20/40/60/80% = 0.325 / 0.728 / 1.109 / 1.603; dlh_norm cuts = 0.553 / 0.920 / 1.240 / 1.652. Low/High regime variants are in the cell output above.

**2. Joint structure:** both invasions occur on 87.8% of days, so Trigger A and Trigger B both fire almost daily - first-trigger priority and max-1-trade/day are essential, and Exit B (opposite touch) would fire ~88% of the time and should be dropped. The joint table shows **dhl (downward invasion) dominates**: dhl Q1 x dlh Q5 still gives P(green) = 0.868 and dhl Q5 x dlh Q1 gives 0.224 - the upward trigger adds almost nothing once dhl is known. Same-quintile conflicts occur on only ~10% of days.

**3. First-touch reality check (minute data, 2024-10 to 2026-09, 446 days):** 76% of first touches occur in the first hour, so quintile classification at first touch is coarse: at a downward touch dhl_running = 1 + (gap below yesterday's low)/R_t by construction, and Q1/Q2 can never be observed at a downward touch. The relation at first touch is **U-shaped, not monotonic**: moderate break (Q3, n=111) -> P(green) = 0.306 (strong short); extreme washout (Q5, n=32) -> 0.562 (bounce). Late up-touches are strongly bullish (11-13h: 0.889, 13-15h: 0.929, small n).

**4. The tradeable form is the time-conditioned checkpoint, and it works:** P(day ends green | running dhl at fixed checkpoints) is perfectly monotonic at 12:00, 13:00, and 14:00, and sharpens into the afternoon:

| dhl_running quintile @14:00 | Q1 | Q2 | Q3 | Q4 | Q5 |
|---|---|---|---|---|---|
| P(green) | 0.667 | 0.610 | 0.547 | 0.405 | 0.307 |

Logit at 14:00: slope -0.804, p < 0.0001, pseudo-R2 = 0.077, n = 499. This validates the rule's core long/short logic in leakage-free, tradeable form: **by ~14:00, small invasion -> long, large invasion -> short**, with 2 hours left to the close. The "no trades after 2:30 PM" constraint is supported.

**Revisions to the rule before implementation:** (a) replace Exit B with end-of-day or stop-based exits (opposite touch fires too often); (b) add a washout override - at a downward *touch* with dhl_running in Q5, the bounce probability (0.562) contradicts the short signal, so either skip the trade or wait for the 14:00 checkpoint to re-classify; (c) the Q1/Q2 long signal is a *non-touch* signal (yesterday's low held through the checkpoint), not a touch-triggered one; (d) walk-forward re-estimation of the quintile cuts is required - the thresholds above are estimated on full history and the checkpoint study covers only two years.


In [ ]:
qb = QuantBook()
equity = qb.add_equity("SPY")

In [ ]:
# Diagnostic: recompute the checkpoint metric for 2007-05-31 exactly as the backtest should
qqq_h = qb.add_equity("QQQ", Resolution.HOUR)

daily = qb.history(qqq.symbol, datetime(2007, 5, 24), datetime(2007, 6, 2), Resolution.DAILY).droplevel(0)
print("Daily bars:")
print(daily[["open", "high", "low", "close"]])

hours = qb.history(qqq_h.symbol, datetime(2007, 5, 31), datetime(2007, 6, 1), Resolution.HOUR)
print("\nRaw history frame index levels:", hours.index.names)
if len(hours.index.names) > 1:
    hours = hours.loc[qqq_h.symbol]
print("\nHour bars 2007-05-31:")
print(hours[["open", "high", "low", "close"]])

prev = daily.loc[daily.index < pd.Timestamp("2007-05-31")].iloc[-1]
through_14 = hours[hours.index.time <= datetime(2007, 5, 31, 14, 0).time()]
running_low = through_14["low"].min()

prev_high = float(prev["high"])
prev_range = float(prev["high"] - prev["low"])
dhl = (prev_high - running_low) / prev_range
print(f"\nprev_high={prev_high:.2f} prev_low={float(prev['low']):.2f} prev_range={prev_range:.2f}")
print(f"running_low_through_14={running_low:.2f}")
print(f"dhl_running should be: {dhl:.3f}   (backtest logged 1.000)")


### Section 11. Backtest validation (run 2026-09-30)

The 14:00 checkpoint rule was implemented in `main.py` and run over 2007-01-01 to 2026-09-30 on QQQ hourly bars (13:00-14:00 hour bar provides the exact checkpoint), VIX daily for regime conditioning, walk-forward quintile cuts (trailing 504 observations, regime-conditional when >= 100 samples), ATR(14) stop, EOD liquidation at 15:45, max 1 trade/day.

**Implementation bug caught and fixed (run 1 discarded):** subscribing QQQ at both HOUR and DAILY resolution collides bars in Slice.bars (same Symbol key); the fill-forwarded daily bar leaked the prior day's low into the running-low tracker, pinning dhl_running at exactly 1.000 on most days and permanently blocking the Q5 short. Fix: single hourly subscription with daily stats rolled manually at day boundaries. Verified against raw history: correct dhl for 2007-05-31 is -0.012, run 1 computed 1.000.

**Run 2 (correct) results, $100k initial, no leverage:**

| Metric | Value |
| --- | --- |
| Net profit | +15.96% total (0.76% CAGR) |
| Total fees | $36,952 (37% of initial equity) |
| Win rate / P-L ratio | 52% / 0.98 |
| Expectancy | +0.025% per trade |
| Sharpe / Alpha vs QQQ | -0.41 / -0.015 |
| Max drawdown | 18.6% |
| Beta / Annual vol | 0.005 / 3.6% (mostly in cash) |

**Verdict:** the checkpoint signal is structurally confirmed out-of-sample (fires per spec, gross edge positive: ~53% gross over 20 years before fees), but the edge is too small for its turnover - costs consume roughly 70% of gross profit. Net risk-adjusted performance is far below buy-and-hold QQQ. Directions that could rescue it: trade only the strongest cells (Q1 longs, Mid-regime only), cut turnover/fees (passive entries, wider conviction thresholds), or use the signal as a sizing overlay on an existing book rather than a standalone strategy. Per-trade expectancy (+2.5 bps) is an order of magnitude below the research daily-direction hit rates (66.7% vs 30.7%) implied - the day-direction edge does not translate one-for-one into a 14:00-to-close P&L edge because a large share of the daily move has already happened by 14:00.


In [ ]:
# Section 12a: replicate the backtest trade stream from raw data for gain-chart analysis
qqq_h = qb.add_equity("QQQ", Resolution.HOUR)
hours_all = qb.history(qqq_h.symbol, datetime(2007, 1, 1), datetime(2026, 9, 30), Resolution.HOUR)
hours_all = hours_all.loc[qqq_h.symbol]
hours_all.index = pd.to_datetime(hours_all.index.date) + pd.to_timedelta(hours_all.index.hour, unit="h")
print("Hour bars:", len(hours_all), "| range:", hours_all.index.min(), "to", hours_all.index.max())

vix_daily = vix_df["vix"]  # from cell 2 (calendar-dated VIX closes)
print("VIX rows:", len(vix_daily))


In [ ]:
# Section 12b: faithful replication of the main.py rule -> per-trade return series
fallback_cuts = [0.325, 0.728, 1.109, 1.603]
regime_weight = {"low": 0.87, "mid": 1.00, "high": 0.97}

dhl_history = []
true_ranges = []
prev_high = prev_low = prev_close = None
atr = None
trade_rows = []

for day, day_bars in hours_all.groupby(hours_all.index.date):
    # finalize the session that just completed (mirrors _finalize_daily)
    if prev_close is not None:
        day_high = float(day_bars["high"].max())
        day_low = float(day_bars["low"].min())
        tr = max(day_high - day_low, abs(day_high - prev_close), abs(day_low - prev_close))
        true_ranges.append(tr)
        if len(true_ranges) > 14:
            true_ranges.pop(0)
        if len(true_ranges) == 14:
            atr = float(np.mean(true_ranges))
    # NOTE: this block finalizes the PREVIOUS session, so capture before overwriting
    if prev_high is not None and atr is not None:
        cp = day_bars[day_bars.index.hour == 14]
        if len(cp) == 1:
            running_low = float(day_bars[day_bars.index.hour <= 14]["low"].min())
            prev_range = prev_high - prev_low
            if prev_range > 0:
                dhl_running = (prev_high - running_low) / prev_range
                vix_prev = vix_daily.shift(1).asof(pd.Timestamp(day))
                regime = "low" if vix_prev < 15 else ("high" if vix_prev > 25 else "mid")
                if len(dhl_history) < 100:
                    cuts = fallback_cuts
                else:
                    recent = dhl_history[-504:]
                    rv = [v for v, r in recent if r == regime]
                    vals = rv if len(rv) >= 100 else [v for v, _ in recent]
                    cuts = list(np.percentile(vals, [20, 40, 60, 80]))
                q = 5
                for i, cut in enumerate(cuts):
                    if dhl_running <= cut:
                        q = i + 1
                        break
                dhl_history.append((dhl_running, regime))
                if q in (1, 2, 5):
                    entry = float(cp["close"].iloc[0])
                    exit_px = float(day_bars["close"].iloc[-1])
                    direction = 1 if q in (1, 2) else -1
                    trade_rows.append({
                        "date": pd.Timestamp(day),
                        "quintile": q,
                        "direction": "long" if direction == 1 else "short",
                        "regime": regime,
                        "dhl": dhl_running,
                        "ret": direction * (exit_px / entry - 1.0),
                        "weight": regime_weight[regime],
                    })
    # roll today's completed session into prev stats for the next iteration
    prev_high = float(day_bars["high"].max())
    prev_low = float(day_bars["low"].min())
    prev_close = float(day_bars["close"].iloc[-1])

trades = pd.DataFrame(trade_rows).set_index("date")
print("Trades replicated:", len(trades), "(backtest round trips: 2935)")
print(trades["direction"].value_counts().to_string())
print("\nMean gross trade return (bps):", round(trades["ret"].mean() * 1e4, 2))
print("Hit rate:", round((trades["ret"] > 0).mean(), 3))


In [ ]:
# Section 12c: gain chart - cumulative weighted trade return by trade sequence
trades_sorted = trades.sort_index()
trades_sorted["weighted_ret"] = trades_sorted["ret"] * trades_sorted["weight"]
trades_sorted["cum_gain"] = trades_sorted["weighted_ret"].cumsum()
trades_sorted["trade_no"] = np.arange(1, len(trades_sorted) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(trades_sorted["trade_no"], trades_sorted["cum_gain"], lw=1.2, color="tab:blue")
axes[0].axhline(0, color="black", lw=0.8, ls="--")
axes[0].set_title("Gain chart - all trades (weighted)")
axes[0].set_xlabel("Trade number")
axes[0].set_ylabel("Cumulative return (decimal)")

for regime, grp in trades_sorted.groupby("regime"):
    axes[1].plot(np.arange(1, len(grp) + 1), grp["weighted_ret"].cumsum(), lw=1.2, label=regime)
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set_title("Gain by VIX regime")
axes[1].set_xlabel("Trade number")
axes[1].set_ylabel("Cumulative weighted return")
axes[1].legend()

for direction, grp in trades_sorted.groupby("direction"):
    axes[2].plot(np.arange(1, len(grp) + 1), grp["weighted_ret"].cumsum(), lw=1.2, label=direction)
axes[2].axhline(0, color="black", lw=0.8, ls="--")
axes[2].set_title("Gain by direction")
axes[2].set_xlabel("Trade number")
axes[2].set_ylabel("Cumulative weighted return")
axes[2].legend()
plt.tight_layout()
plt.show()

print("\nPer-regime stats (weighted returns, bps):")
for regime, grp in trades_sorted.groupby("regime"):
    print(f"  {regime:4s}: n={len(grp):4d}  mean={grp['weighted_ret'].mean()*1e4:6.2f}  sum={grp['weighted_ret'].sum()*1e4:9.1f}  hit={(grp['weighted_ret']>0).mean():.3f}")
print("\nPer-direction stats (weighted returns, bps):")
for direction, grp in trades_sorted.groupby("direction"):
    print(f"  {direction:5s}: n={len(grp):4d}  mean={grp['weighted_ret'].mean()*1e4:6.2f}  sum={grp['weighted_ret'].sum()*1e4:9.1f}  hit={(grp['weighted_ret']>0).mean():.3f}")
print("\nTotal weighted gain (bps):", round(trades_sorted["weighted_ret"].sum() * 1e4, 1))


## Section 12d: Gain Chart Findings and Recommendations

The trade-stream replication (Section 12b) recovered **2,974 trades vs. 2,935 backtest round trips** (~1.3% gap, acceptable for offline analysis) with a mean gross trade return of **+0.97 bps** and hit rate of **52.3%**.

### Gain chart decomposition (Section 12c)

| Segment | n | Mean (bps) | Total (bps) | Hit rate |
|---|---|---|---|---|
| VIX regime: low (<15) | 936 | -0.54 | -507 | 0.485 |
| VIX regime: mid (15-25) | 1,505 | +1.97 | +2,961 | 0.544 |
| VIX regime: high (>25) | 533 | +0.91 | +486 | 0.531 |
| Long | 1,974 | +2.01 | +3,976 | 0.552 |
| Short | 1,000 | -1.04 | -1,036 | 0.467 |

### Interpretation

1. **The short side destroys value.** Cumulative weighted gain for shorts is negative (-1,036 bps) across 1,000 trades. The long book (+3,976 bps) fully funds it, but removing or gating shorts would have improved the total gain from ~2,939 bps to ~3,976 bps, a ~35% uplift.
2. **The low-VIX regime weight (0.87) is not a sufficient handicap.** Even after the 0.87 multiplier, low-regime trades lose -507 bps in aggregate. The weight scale implies these trades should have been excluded entirely, not merely down-weighted.
3. **Mid-VIX is the edge.** The bulk of the strategy's profit (+2,961 bps of +2,939 bps total) comes from the mid regime with the highest hit rate (54.4%) and the best mean per-trade return.
4. **Gain-chart shape:** the all-trades curve (panel 1) is monotonically rising with no prolonged flat stretches, which suggests the signal does not decay materially over the 2007-2026 sample; the mid-regime curve rises smoothly, while the low-regime curve bleeds steadily.

### Recommended next steps for the backtest agent

- Test long-only variants; if the short leg is retained, gate it to the mid/high regimes only.
- Replace the low-regime 0.87 weight with a hard filter and re-run the walk-forward cut optimization without low-regime observations.
- Verify the -1.3% trade-count gap (2,974 vs 2,935) is fill-timing related rather than a logic divergence before deploying any variant.
